# Langfuse 03 · 标注与数据（Annotation Queue ↔ Dataset）

课案「标注」与「数据」两节**只给了 Web UI 截图、没给代码**。本 notebook 把这些截图
背后的 REST API 补全成可跑的代码，并讲清这两页在整个评估闭环里各自站哪个位置：

```mermaid
graph LR
    A["自动打分<br/>04 / 05 / 06"] --> B["找出可疑样本<br/>（指标偏低的那批 trace）"]
    B --> C["人工标注<br/>Annotation Queue（本节）"]
    C --> D["沉淀成数据集<br/>Dataset（本节）"]
    D --> E["拿数据集回归评估<br/>04 / 05 / 06 循环"]
    E --> A
```

一句话记住分工：

| 环节 | 负责什么 | 给不出什么 |
|---|---|---|
| 自动打分（04/05/06） | **大规模找问题**：哪条不对劲 | 给不出「正确答案是什么」 |
| 人工标注（本节前半） | **给出标准答案**（ground truth） | 不能批量、费人力 |
| 数据集（本节后半） | **把答案固化下来**，下次改动拿它回归 | 不会自己发现新问题 |

> **本 notebook 由 `Agent/06_langfuse/` 下 1 个脚本构成**：
> `07_标注与数据_jxsd.py`（新增文件，421 行）。
> 它是「监控与评估 → 评估 → 标注」（四张截图）与「监控与评估 → 数据」（三张截图）
> 这两节课案的**代码补全**：课案只截了 UI，这里把 UI 背后的 API 走一遍。

**官方文档**
- 人工标注（Annotation）：<https://langfuse.com/docs/evaluation/annotation>
- 数据集与实验（Datasets）：<https://langfuse.com/docs/evaluation/experiments/datasets>
- 评估总览：<https://langfuse.com/docs/evaluation/overview>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 会真的连本机 Langfuse（`http://localhost:3001`），并调用大模型 |
| 依赖 | `langfuse`（SDK v4.15.1）/ `langchain` / `langchain-openai`（venv 已装） |
| 密钥 | `settings.langfuse_public_key` / `langfuse_secret_key`（已配置）、`settings.api_key` |
| 前置服务 | 本机 Langfuse：`http://localhost:3001`（**v4 的 events_only 模式**） |
| 预计耗时 | 约 1 分钟（其中大部分是数据集运行时逐条调用大模型） |

**服务没起来会怎样？** 不会崩。第 0.1 节会把「密钥在不在、端口通不通」先打出来；
密钥为空时全篇走**降级路径**：每一步只打印「本应发送的 REST 报文」，
用一个本地桩对象接住返回值，让整条链路的写法仍然完整可读。

课案「安装」一节给的四步（想从零起一个 Langfuse 时照它做）：

1. `git clone https://github.com/langfuse/langfuse.git` → `cd langfuse` → `docker compose up -d`，
   起来后访问 <http://localhost:3000>；
2. 首次注册的账号即为管理员；新建项目 → `Settings → API Keys` → 创建密钥；
3. 把密钥写进 `F:\ProGram\Python_Base\.env`：`LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` /
   `LANGFUSE_HOST`（本地 docker 部署写 `http://localhost:3000`）；
4. 重跑本 notebook，去 **Human Annotation** 与 **Datasets** 两个页面看结果。

## 本节地图

```mermaid
graph TD
    A["1. 客户端初始化<br/>LANGFUSE_READY + api_call 统一封装"] --> B["2. 开场：本机 Langfuse 状态"]
    B --> C["3. 标注（一）<br/>Score Config：定义打分维度"]
    C --> D["4. 标注（二）<br/>建队列 → 放 trace → 指派标注员"]
    D --> E["5. 标注（三）<br/>把人工分取回来"]
    E --> F["6. 数据（一）<br/>标注结论 → Dataset"]
    F --> G["7. 数据（二）<br/>Dataset Run：跑一遍，逐条记输出与分数"]
    G --> H["8. 收尾：闭环再说一遍"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 小节 | 对应源文件的功能 | 对应 Langfuse 的 REST 路径 |
|---|---|---|
| 1 | `LANGFUSE_READY` + `api_call()` | —— |
| 2 | `__main__` 开场的两段打印 | —— |
| 3 | `create_score_configs()` | `POST /api/public/score-configs` |
| 4 | `build_annotation_queue()` | `GET .../annotation-queues`、`POST .../{id}/items`、`POST .../{id}/assignments` |
| 5 | `read_queue()` | `GET .../annotation-queues/{id}/items`、`GET /api/public/v2/scores` |
| 6 | `build_dataset_from_annotation()` | `POST /api/public/datasets`、`POST /api/public/dataset-items` |
| 7 | `run_dataset_experiment()` | 本地跑，结果在 UI 的「数据集 → Runs」看 |

**上下游衔接**：

- **上一课** `02_评估与打分.ipynb`（`04/05/06` 三个脚本）负责**自动打分**——本课第 4 节
  等着它产出的「疑似有问题」的 trace（这里用两条模拟 id 走通流程）；
- **下一课**再回到自动评估：把本课沉淀的 `expected_output` 当作判分基准，
  同一个数据集换个 `run_name` 再跑，就能并排对比「改完到底有没有变好」。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课**不往磁盘写任何文件**（标注队列、数据集都在 Langfuse 服务端），
> 所以 `WORKDIR` 只是照抄模板留着的；并发跑同章其它 notebook 不会互相踩文件。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\06_langfuse\tmp_nb_work
```

## 0.1 前置条件自检

本课是 🔴 需外部服务档。这一格只做一件事：**把「本机到底有什么」先打出来**，
这样后面哪一段走了降级路径，你一眼就知道为什么。

检查四样东西：

| 检查项 | 缺了会怎样 |
|---|---|
| `langfuse` 包 | 全篇无法运行（本机 venv 已装 4.15.1） |
| `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` | 全篇走降级：只打印「本应发送的报文」，返回本地桩对象 |
| `LANGFUSE_HOST` 能不能连上 | 连不上 → SDK 每一步都会抛连接错误；密钥在但服务没起时请先起服务 |
| 模型 `api_key` | 第 7 节「数据集运行」跑不了（它要真调一次大模型） |

> **不打印密钥值**，只看「在不在」。

In [ ]:
# 前置条件自检：只打印现状，不抛异常 —— 缺什么，后面走降级分支时你能立刻对上号。
import socket
import urllib.parse

from config import settings

_todo: list[str] = []

try:
    import langfuse

    print(f"[OK] langfuse {getattr(langfuse, '__version__', '?')} 已装 —— 标注队列与数据集 API 可用")
except ImportError:
    _todo.append("langfuse")
    print("[跳过] 缺 langfuse：先 `uv add langfuse`，本 notebook 全篇需要它")

_pk_ok = bool(settings.langfuse_public_key)
_sk_ok = bool(settings.langfuse_secret_key)
print(f"[信息] LANGFUSE_PUBLIC_KEY = {'已配置' if _pk_ok else '未配置'}")
print(f"[信息] LANGFUSE_SECRET_KEY = {'已配置' if _sk_ok else '未配置'}")
print(f"[信息] LANGFUSE_HOST       = {settings.langfuse_host}")

_parsed = urllib.parse.urlparse(settings.langfuse_host)
_lf_host = _parsed.hostname or "localhost"
_lf_port = _parsed.port or (443 if _parsed.scheme == "https" else 80)
try:
    with socket.create_connection((_lf_host, _lf_port), timeout=2.0):
        print(f"[OK] {_lf_host}:{_lf_port} 已连通 —— 标注队列与数据集会真的写进这台 Langfuse")
except OSError as exc:
    print(f"[跳过] {_lf_host}:{_lf_port} 连不上（{type(exc).__name__}）—— "
          "若密钥已配置，请先把 Langfuse 起起来")

print(f"[信息] 模型 {settings.model_name} 的 api_key = "
      f"{'已配置' if settings.api_key else '未配置'}（第 7 节数据集运行要真调它）")

if not (_pk_ok and _sk_ok):
    print("[跳过] Langfuse 密钥为空：全篇走「打印 REST 报文 + 本地桩对象」的降级路径")
if _todo:
    print(f"[跳过] 缺少：{_todo}")

### 预期输出

本机（Langfuse 已配好、3001 端口在听）会看到：

```text
[OK] langfuse 4.15.1 已装 —— 标注队列与数据集 API 可用
[信息] LANGFUSE_PUBLIC_KEY = 已配置
[信息] LANGFUSE_SECRET_KEY = 已配置
[信息] LANGFUSE_HOST       = http://localhost:3001
[OK] localhost:3001 已连通 —— 标注队列与数据集会真的写进这台 Langfuse
[信息] 模型 deepseek-flash 的 api_key = 已配置（第 7 节数据集运行要真调它）
```

密钥没配时会换成这两行（后面整篇变成「打印报文」的降级演示）：

```text
[信息] LANGFUSE_PUBLIC_KEY = 未配置
[信息] LANGFUSE_SECRET_KEY = 未配置
[跳过] Langfuse 密钥为空：全篇走「打印 REST 报文 + 本地桩对象」的降级路径
```

## 1. 客户端初始化：先判断密钥是否就绪

源文件里的第一条设计：**把「有没有密钥」这一步提到最前面**，用一个模块级常量
`LANGFUSE_READY` 记住结论。后面每个 API 调用都只看这个常量，
于是「真连服务」与「只打印报文」两条路径**共用同一份业务代码**，
不会写成两套互相抄错的分支。

| 情况 | `langfuse` 变量 | 后面每一步 |
|---|---|---|
| 密钥齐全 | `Langfuse(...)` 实例 | 真的发请求，返回服务端的对象 |
| 密钥为空 | `None` | 打印「本应发送的 POST 报文」，返回本地 `SimpleNamespace` 桩对象 |

同一格里还要把**大模型**准备好：第 7 节的「数据集运行」要真跑一次系统。

> 注意 `host=settings.langfuse_host` 传的是本机的 `http://localhost:3001`
> （v4 的 events_only 部署），不是官方云的 `https://cloud.langfuse.com`。

In [ ]:
import json
from types import SimpleNamespace

from langchain.chat_models import init_chat_model
from langfuse import Langfuse
# langfuse 客户端只在本文件里用；标注队列与数据集都通过它的 .api.* 调用。
from config import settings

# ---------- 0. 客户端初始化：先判断密钥是否就绪 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        # 密钥为空置 None：下面所有 api_call 会走降级分支，打印报文并返回本地桩对象。
        host=settings.langfuse_host,
    )
else:
    langfuse = None

    # 本文件要真跑一次大模型产生「待标注样本」，所以模型必须就绪。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

## 2. `api_call()`：真调用与降级打印**共用同一个出口**

这是全篇最关键的一层薄封装。它收两组参数：

| 参数 | 真连上时用 | 降级时用 |
|---|---|---|
| `endpoint` | 不用，只为了打印时能说清「打的是哪条路径」 | 拼进 `[降级] 本应调用 POST …` 这行 |
| `func` | SDK 里对应的方法（如 `langfuse.api.score_configs.create`） | 传 `None`，不会被调用 |
| `fallback_id` | 不用 | 桩对象的 `id`，让上层「拿 id 继续往下走」的代码不变 |
| `**payload` | 原样展开成 SDK 的关键字参数 | `json.dumps` 打印出来，就是 SDK 的请求体 |

有了它，**同一个 `create_score_configs()` 既能真跑，也能在没服务时把要发的报文
原样打出来**——把报文里的 endpoint 与 payload 抄进 Postman / curl 就能手工复现。

In [ ]:
def api_call(endpoint: str, func, fallback_id: str, **payload):
    """统一封装一次 REST 调用。

    真连上 Langfuse → 调 SDK 对应方法；
    密钥为空       → 打印「本应发送的报文」，并返回一个带 id 的本地桩对象，
                     这样上层代码拿 id 继续往下走的逻辑完全一致。
    """
    if LANGFUSE_READY:
        return func(**payload)
    print(f"    [降级] 本应调用  POST {endpoint}")
    print("           payload = " + json.dumps(payload, ensure_ascii=False, default=str))
    return SimpleNamespace(id=fallback_id, **payload)

## 3. 开场：本机 Langfuse 状态与密钥配置指引

源文件 `__main__` 的头两段打印。它的作用不是「跑逻辑」，而是**先把读者扶到正确的
心里位置**：密钥没配时先讲清怎么配，并明确告诉大家「不配也能跑完这一篇」——
免得学员误以为「必须先装好 Langfuse 才能跟着做」。

| 分支 | 打印什么 |
|---|---|
| 密钥为空 | 四步安装指引 + 「下面走降级演示」的说明 |
| 密钥已配 | 一行「标注与数据集将写入 <host>」 |

In [ ]:
# 入口：先讲清 Langfuse 怎么配，再走「打印报文 + 真跑一次」的降级流程。
print("=" * 72)
print("标注（Annotation Queue）与数据（Dataset）—— 人工评估与回归测试集")
print("=" * 72)

if not LANGFUSE_READY:
    print("\n【进入降级演示】Langfuse 密钥为空")
    print("  settings.langfuse_public_key = '' ，settings.langfuse_secret_key = ''")
    print("-" * 72)
    # 按课案「安装」一节的顺序给出四步，照着做就能看到真实的标注队列与数据集页面。
    print("要看到真实的标注队列与数据集页面，按课案「安装」一节准备环境：")
    print("  1) git clone https://github.com/langfuse/langfuse.git")
    print("     cd langfuse")
    print("     docker compose up -d          # 启动后访问 http://localhost:3000")
    print("  2) 首次注册的账号即为管理员；新建项目 → Settings → API Keys → 创建密钥")
    print("  3) 写入 F:\\ProGram\\Python_Base\\.env ：")
    print("        LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("        LANGFUSE_SECRET_KEY=sk-lf-...")
    print("        LANGFUSE_HOST=http://localhost:3000    # 本地 docker 部署用这个")
    print("  4) 重新运行本脚本，去 Human Annotation / Datasets 两个页面看结果")
    print("-" * 72)
    # 上面讲「怎么配」，下面讲「不配也能跑」—— 学员不会误以为必须先配好才能运行。
    print("下面不依赖 Langfuse 服务：把每一步的 REST 报文打印出来，")
    print("并真跑一次大模型产生「待标注的样本」和「数据集运行」记录。")
    print("=" * 72)
else:
    print("Langfuse 已配置，标注与数据集将写入：", settings.langfuse_host)

### 预期输出

密钥已配好时只有一行（本机就是这一种）：

```text
========================================================================
标注（Annotation Queue）与数据（Dataset）—— 人工评估与回归测试集
========================================================================
Langfuse 已配置，标注与数据集将写入： http://localhost:3001
```

密钥为空时会多出一整段中文配置指引（整篇随后变成降级演示）：

```text
【进入降级演示】Langfuse 密钥为空
  settings.langfuse_public_key = '' ，settings.langfuse_secret_key = ''
------------------------------------------------------------------------
要看到真实的标注队列与数据集页面，按课案「安装」一节准备环境：
  1) git clone https://github.com/langfuse/langfuse.git
…
下面不依赖 Langfuse 服务：把每一步的 REST 报文打印出来，
并真跑一次大模型产生「待标注的样本」和「数据集运行」记录。
```

## 4. 标注（一）：定义打分维度（Score Config）

人工标注不能「随便打个分」。**先定义打分维度**，再把它挂到队列上，
标注员进队列时看到的就是这几个固定的表单字段 —— 这样不同人打出来的分才可比。

打分维度（Score Config）的三种类型：

| data_type | 用途 | 必填项 | 例子 |
|---|---|---|---|
| `NUMERIC` | 连续评分 | `min_value` / `max_value` | 准确性 0~1、满意度 1~5 |
| `CATEGORICAL` | 单选分类 | `categories`（每项含 `value`(数字) + `label`） | 问题类型：咨询/投诉/其他 |
| `BOOLEAN` | 是/否 | 无 | 回答是否有害、是否答非所问 |

> ⚠️ **本课最值钱的一个坑**：`CATEGORICAL` 的 `categories[].value` **必须是数字**。
> SDK 的 `ConfigCategory.value` 类型是 `float`，写成 `"consult"` 这种字符串会被
> 服务端拒掉：
>
> ```text
> Category must be an array of objects with label value pairs,
> where labels and values are unique.
> ```
>
> `label` 才是给人看的显示名，`value` 是给程序比对的数字 —— **两者分工别搞反**。

In [ ]:
# ---------- 1. 定义打分维度（Score Config） ----------
def create_score_configs() -> dict:
    """先把「人工要按什么标准打」定义出来，后面建队列时要挂上这些维度。"""
    print("\n" + "=" * 72)
    print("第一步：定义打分维度 POST /api/public/score-configs")
    print("=" * 72)

    configs = {}

    # ① 数值型：人工给 0~1 的连续分
    configs["回答准确性"] = api_call(
        "/api/public/score-configs",
        (langfuse.api.score_configs.create if LANGFUSE_READY else None),
        "cfg-accuracy-001",
        name="回答准确性",
        data_type="NUMERIC",
        # NUMERIC 必须给 min/max：UI 上才会渲染成滑杆，也才能校验人工输入的范围。
        min_value=0.0,
        max_value=1.0,
        description="人工判断：回答是否正确解决了用户问题（0=完全错误，1=完全正确）",
    )

    # ② 布尔型：是/否，用于红线检查
    configs["回答是否有害"] = api_call(
        "/api/public/score-configs",
        (langfuse.api.score_configs.create if LANGFUSE_READY else None),
        "cfg-harmful-001",
        name="回答是否有害",
        # BOOLEAN 不用额外字段；适合「是/否」的红线检查（是否有害、是否答非所问）。
        data_type="BOOLEAN",
        description="回答中是否包含有害/违规内容",
    )

    # ③ 分类型：给样本归类，方便后续按类型切片分析
    configs["问题类型"] = api_call(
        "/api/public/score-configs",
        (langfuse.api.score_configs.create if LANGFUSE_READY else None),
        "cfg-category-001",
        name="问题类型",
        data_type="CATEGORICAL",
        categories=[
            # ⚠️ 实测踩坑：CATEGORICAL 的 **value 必须是数字**，不能写字符串。
            #    SDK 的 `ConfigCategory.value` 类型是 `float`，写成 "consult" 这种
            #    字符串会被服务端拒掉：
            #      Category must be an array of objects with label value pairs,
            #      where labels and values are unique.
            #    label 才是给人看的显示名 —— 两者分工别搞反。
            {"value": 0, "label": "咨询"},
            {"value": 1, "label": "投诉"},
            {"value": 2, "label": "其他"},
        ],
        description="给用户问题归类，便于分桶看指标",
    )

    for name, cfg in configs.items():
        print(f"  ✓ {name}  id={cfg.id}  data_type={cfg.data_type}")
    return configs

三个维度定义好后**立刻调用一次**（原来在 `__main__` 里的那一行）。
返回的 `configs` 要留给第 5 节「建队列」用 —— 队列要挂上这几个维度的 id。

> 这一格**不是幂等**的：Langfuse 允许同名 Score Config 存在多份，重跑会在服务端
> 再建三个新维度（id 每次都不同）。所以「预期输出」里的 id 是你的机器上的值，
> 不必和下面完全一致；真正要固定下来的是 `data_type` 这三个字。

In [ ]:
configs = create_score_configs()

### 预期输出

```text

========================================================================
第一步：定义打分维度 POST /api/public/score-configs
========================================================================
  ✓ 回答准确性  id=4ee08779-7b3c-4c37-b6f7-4aab53bd857c  data_type=NUMERIC
  ✓ 回答是否有害  id=68ab7add-37fa-456b-847a-ea969533fd18  data_type=BOOLEAN
  ✓ 问题类型  id=af806a95-7949-407b-b802-707c8fccc561  data_type=CATEGORICAL
```

`data_type` 是**服务端按我们传的字符串回填的枚举**（`ScoreConfigDataType.NUMERIC` 这类，
打印时显示成普通字符串）。三个 id 写入服务端后，去 `Settings → Score Configs` 就能看到。

> ⚠️ 上面三行的 `id` 是 Langfuse 每次新建维度时分配的 **UUID，每次运行都不同**；
> 稳定的是「名称 + `data_type`」这三对以及创建顺序。正文里那三串只是我这次跑出来的实测值，
> 不必和你的输出逐字一致。

## 5. 标注（二）：建队列 → 放样本 → 指派标注员

这一步是「标注」页四张截图背后的全部 API，一共三个动作：

| 步骤 | 干什么 | 对应 API |
|---|---|---|
| 2 | 建一个标注队列，挂上打分维度 | `api.annotation_queues.create_queue` |
| 3 | 把要复核的 trace 放进队列 | `api.annotation_queues.create_queue_item` |
| 4 | 把队列指派给标注员 | `api.annotation_queues.create_queue_assignment` |

（第 1 步是上一节的 Score Config，第 5 步「人在 UI 上逐条打分」是 Web 操作，
第 6 步取分在下一节。）

这里藏了**两个实测踩出来的坑**，都在下面的代码里就地兜住了：

1. **队列不允许同名重建** —— 再建会 400 `A queue with this name already exists.`。
   本文件是可以反复跑的演示脚本，所以**先按名字查一遍，已有就直接复用**；
   否则第二次跑必崩，而「重跑一次」在演示 / 回归里恰恰是最常见的动作。
2. **指派标注员的 `user_id` 必须是项目里真实存在的成员 id** —— 随便填一个
   （如 `user_A`）会 404 `User not found or not authorized for this project`。
   这是服务端的**引用完整性校验**。真实 id 在控制台 `Settings → Members` 看；
   代码里用 `try/except` 兜住并打印中文说明，因为队列与样本此时都已经建好了。

In [ ]:
# ---------- 2. 建标注队列 / 放样本 / 指派标注员 ----------
QUEUE_NAME = "agent-回答质量复核"

In [ ]:
def build_annotation_queue(configs: dict, trace_ids: list[str]) -> str:
    print("\n" + "=" * 72)
    print("第二~四步：建队列 → 放 trace → 指派标注员")
    print("=" * 72)

    # 1) 建队列，把三个打分维度挂上去（标注员进队列看到的表单就是这几个维度）
    # queue 的 id 要留给后面「放样本」「指派标注员」两步用，所以必须接住返回值。
    #
    # 幂等处理：Langfuse **不允许同名队列** —— 再建会 400
    # `A queue with this name already exists.`（实测踩坑）。
    # 本文件是可以反复跑的演示脚本，所以先按名字查一遍，已有就直接复用；
    # 否则第二次跑必崩，而「重跑一次」在演示/回归里是最常见的动作。
    queue = None
    if LANGFUSE_READY:
        existing = langfuse.api.annotation_queues.list_queues(limit=100)
        queue = next(
            (q for q in getattr(existing, "data", []) if getattr(q, "name", None) == QUEUE_NAME),
            None,
        )
        if queue is not None:
            print(f"  · 已存在同名队列，直接复用：{queue.name}  id={queue.id}")

    if queue is None:
        queue = api_call(
            "/api/public/annotation-queues",
            (langfuse.api.annotation_queues.create_queue if LANGFUSE_READY else None),
            "queue-local-001",
            name=QUEUE_NAME,
            description="把 accuracy 低的 trace 挑出来，人工复核后给出标准答案，沉淀成数据集",
            score_config_ids=[cfg.id for cfg in configs.values()],
        )
        print(f"  ✓ 队列已建：{queue.name}  id={queue.id}")

    # 2) 逐条把要复核的 trace 放进队列（object_type 还能是 OBSERVATION / SESSION）
    for trace_id in trace_ids:
        item = api_call(
            "/api/public/annotation-queues/{queueId}/items",
            (langfuse.api.annotation_queues.create_queue_item if LANGFUSE_READY else None),
            f"queue-item-{trace_id}",
            queue_id=queue.id,
            object_id=trace_id,
            object_type="TRACE",     # 也可以整条会话（SESSION）或某个 span（OBSERVATION）
            status="PENDING",        # PENDING 待处理 / COMPLETED 已完成
        )
        print(f"  ✓ 已入队：{item.object_id}  status={item.status}")

    # 3) 指派标注员（user_id 是 Langfuse **组织成员**的用户 id，必须是项目里真实存在的成员）
    #    ⚠️ 实测踩坑：随便填一个（如 "user_A"）会被服务端拒掉 ——
    #       404 `User not found or not authorized for this project`。
    #       真实 id 在 Langfuse 控制台 → Settings → Members 里看。
    #       这是服务端的**引用完整性校验**，只有拿到真实 id 才能通过；所以这里兜住不崩：
    #       队列与样本都已经建好了，指派失败不影响本节其余演示。
    try:
        assignment = api_call(
            "/api/public/annotation-queues/{queueId}/assignments",
            (langfuse.api.annotation_queues.create_queue_assignment if LANGFUSE_READY else None),
            "queue-assignment-001",
            queue_id=queue.id,
            # 指派之后，标注员登录 Langfuse 就能在 Human Annotation 页面看到这个队列。
            user_id="user_A",
        )
        print(f"  ✓ 已指派标注员：{assignment.user_id}")
    except Exception as exc:   # noqa: BLE001 —— 填的是示例 id，被服务端拒绝属预期情况
        print(f"  ⚠️ 指派标注员失败（{type(exc).__name__}）：{str(exc)[:110]}")
        print("     user_id 必须是本项目里真实存在的成员 id"
              "（控制台 Settings → Members 查看）。")
        print("     队列与样本都已建好，这一步不影响后续演示。")
    return queue.id

现在调用它。`trace_ids` 在真实场景里来自**上一课自动打分偏低的那批样本**；
这里用两条模拟 id 把流程走通（`trace-low-001` / `trace-low-002`）。

> 队列里会**累积**历史条目 —— 每重跑一次就多两条。所以你的实际输出条数
> 只会比我这里多、不会少。

In [ ]:
# 真实场景里，下面的 trace_ids 来自「自动打分偏低」的那批样本；
# 这里用两条模拟 id 走通流程。
queue_id = build_annotation_queue(configs, ["trace-low-001", "trace-low-002"])

### 预期输出

```text

========================================================================
第二~四步：建队列 → 放 trace → 指派标注员
========================================================================
  · 已存在同名队列，直接复用：agent-回答质量复核  id=cmu55ipzd000rnz07c5qslg1p
  ✓ 已入队：trace-low-001  status=PENDING
  ✓ 已入队：trace-low-002  status=PENDING
  ⚠️ 指派标注员失败（NotFoundError）：headers: {'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-wh
     user_id 必须是本项目里真实存在的成员 id（控制台 Settings → Members 查看）。
    队列与样本都已建好，这一步不影响后续演示。
```

**三个值得停一下看的细节**：

1. 「已存在同名队列，直接复用」——**这就是那个幂等坑被兜住的样子**。删掉那段
   `list_queues` 查询，第二次跑本 notebook 就会在这一行 400 崩掉；
2. 两条 trace 都变成 `PENDING`，去 Langfuse 的 **Human Annotation** 页面就能看到
   这个队列，点进去左侧是待处理条目、右侧是我们在第 4 节定义的三个打分维度；
3. **指派失败是预期结果**，不是 Bug：`user_A` 不是本项目成员，服务端做了引用完整性
   校验。把 `user_id` 换成 `Settings → Members` 里的真实 id 就会打印
   `✓ 已指派标注员：<你的 id>`。异常只截断打印前 110 字符，所以你能看到
   一大串响应头 —— 真正的错误信息在末尾，这也是「打印异常原文」的价值。

> ⚠️ 「指派标注员失败」那一行里 `headers: {...}` 是**服务端返回的响应头，每次运行都不同**
> （键的顺序与内容都会变，而且被截断到 110 字符，所以行尾是断的）。稳定的是失败类型
> `NotFoundError` 和后面那两行中文提示；上面那串只是我这次跑出来的实测值，别逐字比对。

## 6. 标注（三）：把人工分取回来

人在 UI 上打完分，分数会**回流到对应的 trace 上**，之后就能按分数名聚合看板。
取分时要认准一个字段：**`source=ANNOTATION`** ——
它是「人工打的分」与「代码打的分」唯一的区分依据，筛选时必用它。

人工分的长相（下面这格把它固化成常量打印出来）：

| 字段 | 含义 |
|---|---|
| `traceId` | 分数挂在哪条 trace 上 |
| `name` | 对应哪个 Score Config（如「回答准确性」） |
| `value` | 人工填的值（NUMERIC 是浮点、BOOLEAN 是 0/1、CATEGORICAL 是那个数字 `value`） |
| `dataType` | 维度的类型 |
| `source` | `ANNOTATION` = 人工标注队列产出的分 |
| `comment` | 人工写的理由（**这一栏常比分数本身更值钱**） |

> ⚠️ **本机这把 Langfuse 的实测坑**：它是 **v4 的 events_only 模式**，
> `GET /api/public/v2/scores` 这条路径**不存在**：
>
> ```text
> 404 This endpoint is not available on deployments running in Langfuse v4 events_only mode.
> ```
>
> v4 把分数改走事件通道了。**UI 上照常能看到这些分**，程序取数要走 v4 的 metrics 接口。
> 源文件的做法是就地兜住、打印数据结构，而不是让整份演示崩在最后一步 —— 下面照旧。

In [ ]:
# 人工打分在 SDK 侧长什么样：没接服务端、或部署形态不支持取分接口时用它做示例。
EXAMPLE_SCORES = [
    {"traceId": "<trace-id>", "name": "回答准确性", "value": 1.0,
     "dataType": "NUMERIC", "source": "ANNOTATION", "comment": "答对了，但没给出单位"},
    {"traceId": "<trace-id>", "name": "回答是否有害", "value": 0,
     "dataType": "BOOLEAN", "source": "ANNOTATION", "comment": ""},
]


def print_example_scores() -> None:
    """打印「人工分数长什么样」的示例结构（两条降级路径共用）。"""
    print("    人工在 UI 上打完分之后，SDK 侧取到的数据结构形如：")
    print(json.dumps(EXAMPLE_SCORES, ensure_ascii=False, indent=2))

In [ ]:
def read_queue(queue_id: str) -> None:
    """标注做完之后，怎么把结果取回来做分析。"""
    print("\n" + "=" * 72)
    print("第六步：取回人工分数（分数回流到 trace，可按 name 聚合看板）")
    print("=" * 72)

    if LANGFUSE_READY:
        # 队列里各条的状态
        items = langfuse.api.annotation_queues.list_queue_items(queue_id=queue_id)
        for item in getattr(items, "data", []):
            print(f"  {item.object_id}  {item.status}")

        # 按分数名取人工打出来的分（source=ANNOTATION 表示来自标注队列）
        # source=ANNOTATION 是「人工打的分」与「代码打的分」唯一的区分字段，筛选时用它。
        # ⚠️ 实测踩坑：Langfuse **v4 的 events_only 模式**下这条 REST 路径不存在 ——
        #    404 `This endpoint is not available on deployments running in Langfuse v4
        #    events_only mode.`（v4 把分数改走事件通道了）。
        #    这里兜住并打印数据结构：让读者知道「是部署形态不支持这条路径」，
        #    而不是让整份演示崩在最后一步。
        try:
            scores = langfuse.api.scores.get_many(name="回答准确性", limit=10)
        except Exception as exc:   # noqa: BLE001 —— 部署形态差异，不该把演示炸掉
            print(f"  ⚠️ 按分数名取分失败（{type(exc).__name__}）：{str(exc)[:120]}")
            print("     本机 Langfuse 是 v4 events_only 模式，/api/public/v2/scores 不可用；")
            print("     分数在 UI 的 Human Annotation 页面能正常看到，程序取数要走 v4 的 metrics 接口。")
            print_example_scores()
        else:
            for score in getattr(scores, "data", []):
                print(f"  trace={score.trace_id}  {score.name}={score.value}  comment={score.comment}")
        langfuse.flush()
    else:
        print("    [降级] 本应调用  GET /api/public/annotation-queues/{queueId}/items")
        print("    [降级] 本应调用  GET /api/public/v2/scores?name=回答准确性")
        print_example_scores()

In [ ]:
read_queue(queue_id)

### 预期输出

```text

========================================================================
第六步：取回人工分数（分数回流到 trace，可按 name 聚合看板）
========================================================================
  trace-low-002  PENDING
  trace-low-001  PENDING
  （……上面两行重复出现 6 次：队列里的历史条目会累积，重跑一次多两条……）
  ⚠️ 按分数名取分失败（NotFoundError）：headers: {'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-o
     本机 Langfuse 是 v4 events_only 模式，/api/public/v2/scores 不可用；
     分数在 UI 的 Human Annotation 页面能正常看到，程序取数要走 v4 的 metrics 接口。
    人工在 UI 上打完分之后，SDK 侧取到的数据结构形如：
[
  {
    "traceId": "<trace-id>",
    "name": "回答准确性",
    "value": 1.0,
    "dataType": "NUMERIC",
    "source": "ANNOTATION",
    "comment": "答对了，但没给出单位"
  },
  {
    "traceId": "<trace-id>",
    "name": "回答是否有害",
    "value": 0,
    "dataType": "BOOLEAN",
    "source": "ANNOTATION",
    "comment": ""
  }
]
```

**关于那 12 行 `PENDING`**：真正跑过的机器上，队列里已经有历史条目了，所以打印出来
是 12 行（`trace-low-001/002` 各 6 次）—— 每重跑一次本 notebook 就再多两条，
你看到的行数只会更多。上面用一行说明代替重复，是为了让结构看得清。

**为什么这一段要写 `try/except`**：如果直接在最后一步抛 404，读者会以为
「是我配错了 / 是代码写错了」。兜住之后打印的两行中文把责任说清楚 ——
**是部署形态（v4 events_only）不支持这条路径**，不是配置错。
这也是本课最值得带走的一个习惯：**把「环境差异」与「代码缺陷」用异常处理的措辞区分开**。

## 7. 数据（一）：把标注结论沉淀成数据集

人工标注的价值在于它产出的是**标准答案**：自动指标只能告诉你「哪条不对劲」，
给不出 ground truth。所以下一步就是把标注结论**直接写成 `expected_output` 存进数据集**，
下一轮回归就有了判分基准 —— 这就是课案「标注 → 数据」两节连起来的意思。

| 概念 | 说明 |
|---|---|---|
| **Dataset** | 测试集本身，可版本化 |
| **Dataset Item** | 一条测试项：`input` / `expected_output` / `metadata` |
| **Dataset Run** | 拿这个测试集跑一遍的结果（下一节会产生） |
| **Run Item** | 一次运行里「某条测试项 → 实际输出 + 分数」的对应关系 |

两条黄金测试项就写死在函数里：一条是**可精确判分**的计算题（`15*8+23 = 143`），
一条是**开放问答**（北京天气，期望输出是人工确认的标准答案）。
对照着看这两条，就明白为什么「人工标注」不能被自动指标替代。

In [ ]:
# ---------- 3. 数据：把标注结果沉淀成数据集 ----------
DATASET_NAME = "agent_golden_set"


def build_dataset_from_annotation() -> list:
    """标注过的高价值样本 → 数据集。数据集是回归测试的唯一事实来源。"""
    print("\n" + "=" * 72)
    print("数据：Dataset（测试集）与 Dataset Run（一次实验），对应课案「数据」页")
    print("=" * 72)

    golden_items = [
        # 把标注结论固化成测试项：input 复现问题，expected_output 是人工确认的标准答案。
        {"input": "帮我计算 15 * 8 + 23", "expected_output": "143",
         "metadata": {"来源": "标注队列", "问题类型": "consult"}},
        {"input": "北京今天天气怎么样？", "expected_output": "北京今天晴，气温 30°C，湿度 45%",
         "metadata": {"来源": "标注队列", "问题类型": "other"}},
    ]

    api_call(
        "/api/public/datasets",
        (langfuse.create_dataset if LANGFUSE_READY else None),
        "dataset-local-001",
        name=DATASET_NAME,
        description="人工标注沉淀下来的黄金测试集，用于每次改动的回归评估",
    )
    print(f"  ✓ 数据集已建：{DATASET_NAME}")

    for item in golden_items:
        api_call(
            "/api/public/dataset-items",
            (langfuse.create_dataset_item if LANGFUSE_READY else None),
            "dataset-item-local",
            dataset_name=DATASET_NAME,
            **item,
        )
        print(f"  ✓ 测试项已加入：{item['input']}  →  期望 {item['expected_output']}")

    if LANGFUSE_READY:
        return langfuse.get_dataset(DATASET_NAME).items
    return [SimpleNamespace(**item) for item in golden_items]

调用它，拿到 `dataset_items`。

> ⚠️ 这里有个**会随时间变长的东西**：`langfuse.get_dataset(name).items` 返回的是
> 这个数据集里**当前全部**条目 —— 数据集条目是**追加**的，重跑一次就多两条。
> 所以下一节「数据集运行」会逐条调大模型，条目越多跑得越久。
> 想清空就删掉数据集重建（或用带时间戳的数据集名）。

In [ ]:
dataset_items = build_dataset_from_annotation()

### 预期输出

```text

========================================================================
数据：Dataset（测试集）与 Dataset Run（一次实验），对应课案「数据」页
========================================================================
  ✓ 数据集已建：agent_golden_set
  ✓ 测试项已加入：帮我计算 15 * 8 + 23  →  期望 143
  ✓ 测试项已加入：北京今天天气怎么样？  →  期望 北京今天晴，气温 30°C，湿度 45%
```

注意 `create_dataset` 用同名再调一次**不会报错**（服务端按名字 upsert），
而 `create_dataset_item` 每次都是**新增** —— 这就是上一条提醒的来源。

## 8. 数据（二）：Dataset Run —— 拿数据集跑一遍系统

课案「数据」页那张对比图背后就是这一步：**同一个数据集跑一遍，逐条记录
「实际输出 + 分数」**。它的两个关键字段：

| 字段 | 作用 |
|---|---|
| `datasetItemId` | 这条结果对应哪条测试项 |
| `runName` | **并排对比的钥匙**：换个 `run_name`（换提示词 / 换模型）再跑一次，UI 上就能把两次 run 并排看 |

逐条的打分逻辑是**最朴素的字符串包含**（`expected_output in answer` → 1.0，否则 0.0）。
故意用这么简单的判分，是为了把注意力留给**流程**：真正要讲的不是「怎么打分」，
而是「**怎么把一次实验的每条输出都记下来，供以后对比**」。

> 这一格要真调大模型（本机是 `settings.model_name`），也是整篇最慢的一格。
> 被测系统这里只是裸的 `llm.invoke`，真实项目里换成你的 Agent 流程即可。

In [ ]:
# ---------- 4. 真跑一次「数据集运行」 ----------
def run_dataset_experiment(items) -> None:
    """课案「数据」页那张对比图背后就是这一步：同一个数据集跑一遍，逐条记录输出与分数。"""
    print("\n" + "=" * 72)
    print("Dataset Run：拿数据集跑一遍系统，逐条记录「实际输出 + 分数」")
    print("=" * 72)

    run_items = []
    for item in items:
        answer = llm.invoke(item.input).content            # 被测系统（换成你的 Agent 流程）
        score = 1.0 if str(item.expected_output) in answer else 0.0
        run_items.append({
            "datasetItemId": f"<{item.input[:12]}…>",
            "runName": "baseline-v1",                      # 换个 run_name 再跑一次就能并排对比
            "traceId": "<本次运行的 trace id>",
            "output": answer[:60] + ("…" if len(answer) > 60 else ""),
            "scores": [{"name": "keyword_match", "value": score}],
        })
        # run_name 是并排对比的关键：换个名字再跑一次，两次 run 就能在 UI 上对照。
        print(f"  Q: {item.input}")
        print(f"     实际输出: {answer[:60]}…")
        print(f"     期望输出: {item.expected_output}")
        print(f"     分数: {score}")

    # 把这一轮 run 的每条记录打出来 —— 它就是 UI 上「数据集 → Runs」页看到的内容。
    print("\n  这次运行产生的 dataset run item（UI 上「数据集 → Runs」看到的就是它）：")
    print(json.dumps(run_items, ensure_ascii=False, indent=2))
    print("\n  换个 run_name（比如换个提示词或模型）再跑一遍，两次 run 就能并排对比 ——")
    print("  这就是「改完到底有没有变好」的客观依据，而不是凭感觉说「好像好一点」。")

    if LANGFUSE_READY:
        langfuse.flush()

In [ ]:
run_dataset_experiment(dataset_items)

### 预期输出

大模型的回答每次都不一样，所以**只有结构是稳定的**。下面是本机实跑的输出
（数据集里当时有 8 条，所以「问题 / 答案 / 分数」那三行重复了 4 轮，这里截前两轮）：

```text

========================================================================
Dataset Run：拿数据集跑一遍系统，逐条记录「实际输出 + 分数」
========================================================================
  Q: 北京今天天气怎么样？
     实际输出: 抱歉，我无法实时联网获取北京今天的天气数据，所以不能给你准确的实时温度、降雨和空气质量。

建议你打开手机自带天气 Ap…
     期望输出: 北京今天晴，气温 30°C，湿度 45%
     分数: 0.0
  Q: 帮我计算 15 * 8 + 23
     实际输出: 143…
     期望输出: 143
     分数: 1.0
  ……（上面两轮重复：数据集条目是追加的，这一格对「全部」条目各跑一次）……

  这次运行产生的 dataset run item（UI 上「数据集 → Runs」看到的就是它）：
[
  {
    "datasetItemId": "<北京今天天气怎么样？…>",
    "runName": "baseline-v1",
    "traceId": "<本次运行的 trace id>",
    "output": "抱歉，我无法实时联网获取北京今天的天气数据，所以不能给你准确的实时温度、降雨和空气质量。\n\n建议你打开手机自带天气 Ap…",
    "scores": [
      {
        "name": "keyword_match",
        "value": 0.0
      }
    ]
  },
  {
    "datasetItemId": "<帮我计算 15 * 8 …>",
    "runName": "baseline-v1",
    "traceId": "<本次运行的 trace id>",
    "output": "143",
    "scores": [
      {
        "name": "keyword_match",
        "value": 1.0
      }
    ]
  },
  ……（本机这次共 8 条，其余同上两轮）……
]

  换个 run_name（比如换个提示词或模型）再跑一遍，两次 run 就能并排对比 ——
  这就是「改完到底有没有变好」的客观依据，而不是凭感觉说「好像好一点」。
```

**为什么先问的是天气、后问的是计算题**：`get_dataset().items` 按**新→旧**返回，
而 `golden_items` 里天气那条是后写的，所以它排在最前。**顺序不稳定是正常的**，
要看的是「每条测试项都有自己的输出与分数」这件事本身。

**两处最值得留意的**：

1. 天气那条拿 0 分是**正常的**：本题的判分规则是「期望答案整串出现在回答里」，
   而通用模型不会给出「晴，30°C，湿度 45%」这种具体数值 —— 这正是**数据集+人工标注**
   存在的意义：它把「这条应该答什么」固定下来，才能一眼看出 0 分是**系统能力不足**，
   而不是判分脚本写错了；
2. `runName = "baseline-v1"` 是**写死的**。把它改成 `baseline-v2` 再跑一遍，
   UI 的「数据集 → Runs」就会并排列出两次实验 —— 这一步是「评估」而非「记录」的分界线。

## 9. 收尾：把闭环再说一遍

源文件 `__main__` 的最后两行打印。它其实是这一课的**结论**，所以单独留一格。

In [ ]:
# 四步串起来就是闭环：自动指标找问题 → 人工给答案 → 数据集固化 → 下次改动回归。
print("\n小结：自动指标负责「大规模找问题」，人工标注负责「给出标准答案」，")
print("      数据集负责「把答案固化下来，下次改动拿它回归」——三者缺一不成闭环。")

### 预期输出

```text

小结：自动指标负责「大规模找问题」，人工标注负责「给出标准答案」，
      数据集负责「把答案固化下来，下次改动拿它回归」——三者缺一不成闭环。
```

## 小结

- **标注队列** = 把「自动指标发现的可疑样本」交给人的通道，它产出的是
  **ground truth（标准答案）**，自动指标产不出来；
- **Score Config 必须在建队列之前定义**，并且挂到队列上 —— 标注员看到的表单就是它；
- **`CATEGORICAL` 的 `value` 是数字**（`label` 才是显示名）；
- **队列不允许同名重建**，演示脚本必须先查再复用；**指派标注员要用真实成员 id**；
- **数据集条目是追加的**：`Dataset` 沉淀标准答案，`Dataset Run` 记录一次实验
  的逐条输出与分数，`runName` 就是并排对比的钥匙；
- **v4 events_only 部署下 `/api/public/v2/scores` 不可用**，UI 照常看，程序取数走 metrics 接口。

下一课回到 `02_评估与打分.ipynb` 的循环：把本课的 `expected_output` 当判分基准，
同一数据集换个 `run_name` 再跑 —— 那时候「改完到底有没有变好」就有客观依据了。

## 常见坑

1. **`categories[].value` 写成字符串 → 服务端 400**：
   `Category must be an array of objects with label value pairs, where labels and
   values are unique.`。SDK 的 `ConfigCategory.value` 是 `float`，`label` 才是显示名。
2. **同名队列再建 → 400 `A queue with this name already exists.`**。
   演示脚本要能反复跑，就必须「先 `list_queues` 按名字查、有则复用」。
3. **指派标注员用假的 `user_id` → 404 `User not found or not authorized for this project`**。
   这是引用完整性校验；真实 id 在 `Settings → Members`，代码里要 `try/except` 兜住。
4. **v4 events_only 模式下 `GET /api/public/v2/scores` 不存在**
   （404 `This endpoint is not available on deployments running in Langfuse v4
   events_only mode.`）。分数在 UI 上照常可见，程序取数要走 v4 的 metrics 接口。
5. **数据集条目是追加的**：重跑本 notebook，`get_dataset(...).items` 会越来越长，
   第 8 节随之变慢。这是「数据集可版本化」的另一面，不是 Bug。
6. **异常文本很长**：`str(exc)` 里带着一大串响应头，所以代码里截断到 110/120 字符。
   排查时看**最后一行**才是真正的错误原因。

## 官方链接

- 人工标注 Annotation（本课前半的 UI 与概念）：<https://langfuse.com/docs/evaluation/annotation>
- 数据集 Datasets 与实验：<https://langfuse.com/docs/evaluation/experiments/datasets>
- 评估总览（自动指标 ↔ 人工标注在闭环里各站哪）：<https://langfuse.com/docs/evaluation/overview>
- Score Configs（三种 data_type 的字段定义）：<https://langfuse.com/docs/scores/score-configs>
- REST API 参考（`annotation-queues` / `score-configs` / `datasets` 三组路径）：
  <https://api.reference.langfuse.com/>